# Tahap 4: Case Solution Reuse
**SubCPMK-3 — Penalaran Komputer | NIM: 202310370311358**

Tujuan: Menggunakan putusan lama (top-k kasus termirip) sebagai dasar prediksi solusi untuk kasus baru.

In [ ]:
import re
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

PROCESSED_DIR = Path('../data/processed')
EVAL_DIR      = Path('../data/eval')
RESULTS_DIR   = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load model & data
with open(PROCESSED_DIR / 'tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open(PROCESSED_DIR / 'svm_model.pkl', 'rb') as f:
    svm = pickle.load(f)
with open(PROCESSED_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

df_clean = pd.read_csv(PROCESSED_DIR / 'cases_clean.csv')
df_cases = pd.read_csv(PROCESSED_DIR / 'cases.csv')
df_index = pd.read_csv(PROCESSED_DIR / 'case_index.csv')
X_all = np.load(PROCESSED_DIR / 'tfidf_matrix.npy')

# Buat dictionary solusi: case_id → amar_putusan
case_solutions = dict(zip(df_cases['case_id'], df_cases['amar_putusan']))
case_labels    = dict(zip(df_index['case_id'], df_index['label']))

print(f'Case base dimuat: {len(case_labels)} kasus')
print(f'Label tersedia: {list(set(case_labels.values()))}')

## 4.1 Fungsi Retrieval (dari Tahap 3)

In [ ]:
STOPWORDS_ID = set(stopwords.words('indonesian'))
all_ids = df_index['case_id'].values

def preprocess_query(query):
    query = query.lower()
    query = re.sub(r'[^a-zA-Z\s]', ' ', query)
    tokens = [t for t in query.split() if t not in STOPWORDS_ID and len(t) > 2]
    return ' '.join(tokens)

def retrieve(query: str, k: int = 5):
    """Retrieve top-k kasus termirip (cosine similarity TF-IDF)."""
    q_clean = preprocess_query(query)
    q_vec   = tfidf.transform([q_clean]).toarray()
    sims    = cosine_similarity(q_vec, X_all).flatten()
    top_k   = np.argsort(sims)[::-1][:k]
    return [(all_ids[i], float(sims[i])) for i in top_k]

## 4.2 Ekstrak Solusi dari Top-k Kasus

In [ ]:
def get_top_k_solutions(query: str, k: int = 5):
    """Ambil amar putusan dan label dari top-k kasus termirip."""
    top_k = retrieve(query, k=k)
    solutions = []
    for case_id, score in top_k:
        solutions.append({
            'case_id' : case_id,
            'score'   : score,
            'label'   : case_labels.get(case_id, 'unknown'),
            'amar'    : case_solutions.get(case_id, 'UNKNOWN'),
        })
    return solutions

# Test
q = "terdakwa terbukti membawa narkotika jenis sabu pasal 112"
sols = get_top_k_solutions(q)
for s in sols:
    print(f"{s['case_id']} | score={s['score']:.3f} | label={s['label']}")

## 4.3 Algoritma Prediksi Solusi

In [ ]:
def majority_vote(solutions):
    """Pilih label yang paling banyak muncul di top-k (majority vote)."""
    labels = [s['label'] for s in solutions if s['label'] != 'unknown']
    if not labels:
        return 'unknown'
    return Counter(labels).most_common(1)[0][0]

def weighted_similarity(solutions):
    """Pilih label dengan total similarity terbesar (weighted vote)."""
    weights = {}
    for s in solutions:
        lbl = s['label']
        if lbl == 'unknown':
            continue
        weights[lbl] = weights.get(lbl, 0) + s['score']
    if not weights:
        return 'unknown'
    return max(weights, key=weights.get)

def predict_outcome(query: str, k: int = 5, method: str = 'weighted') -> dict:
    """
    Prediksi amar putusan untuk query kasus baru.
    
    Args:
        query  : Teks deskripsi kasus baru
        k      : Jumlah kasus terdekat
        method : 'majority' atau 'weighted'
    
    Returns:
        dict dengan predicted_label, predicted_solution, top_k_cases
    """
    solutions = get_top_k_solutions(query, k=k)
    
    # Pilih label prediksi
    if method == 'majority':
        pred_label = majority_vote(solutions)
    else:
        pred_label = weighted_similarity(solutions)
    
    # Ambil teks amar dari kasus dengan label sama yang punya score tertinggi
    matching = [s for s in solutions if s['label'] == pred_label]
    pred_solution = matching[0]['amar'] if matching else solutions[0]['amar']
    top5_ids = [s['case_id'] for s in solutions]

    return {
        'predicted_label'   : pred_label,
        'predicted_solution': pred_solution,
        'top_k_cases'       : top5_ids,
        'top_k_scores'      : [s['score'] for s in solutions],
    }

# Uji
result = predict_outcome("terdakwa kepergok membawa sabu 2 gram")
print(f"Prediksi label    : {result['predicted_label']}")
print(f"Prediksi solusi   : {result['predicted_solution'][:200]}")
print(f"Top-5 kasus       : {result['top_k_cases']}")

## 4.4 Demo Manual: 5 Kasus Baru

In [ ]:
demo_queries = [
    {"query_id": "demo_1",
     "query": "terdakwa tertangkap tangan membawa 10 gram sabu jenis metamfetamina"},
    {"query_id": "demo_2",
     "query": "terdakwa menjual dan mengedarkan narkotika golongan 1 kepada beberapa orang"},
    {"query_id": "demo_3",
     "query": "terdakwa menggunakan narkotika untuk diri sendiri sesuai keterangan saksi"},
    {"query_id": "demo_4",
     "query": "terdakwa memiliki dan menyimpan ganja seberat 50 gram dalam rumahnya"},
    {"query_id": "demo_5",
     "query": "terdakwa tidak terbukti terlibat peredaran narkoba berdasarkan bukti yang ada"},
]

predictions = []
for q in demo_queries:
    res = predict_outcome(q['query'])
    predictions.append({
        'query_id'          : q['query_id'],
        'query_text'        : q['query'],
        'predicted_label'   : res['predicted_label'],
        'predicted_solution': res['predicted_solution'][:300],
        'top_5_case_ids'    : ', '.join(res['top_k_cases']),
    })
    print(f"{q['query_id']}: {res['predicted_label']}")

df_pred = pd.DataFrame(predictions)

## 4.5 Simpan Hasil Prediksi

In [ ]:
# Jalankan juga terhadap query evaluasi dari queries.json
with open(EVAL_DIR / 'queries.json', encoding='utf-8') as f:
    eval_queries = json.load(f)

eval_predictions = []
for q in eval_queries:
    res = predict_outcome(q['query_text'])
    eval_predictions.append({
        'query_id'          : q['query_id'],
        'ground_truth_label': q['ground_truth_label'],
        'predicted_label'   : res['predicted_label'],
        'predicted_solution': res['predicted_solution'][:300],
        'top_5_case_ids'    : ', '.join(res['top_k_cases']),
    })

df_eval_pred = pd.DataFrame(eval_predictions)
df_eval_pred.to_csv(RESULTS_DIR / 'predictions.csv', index=False, encoding='utf-8-sig')

print('Saved: data/results/predictions.csv')
display(df_eval_pred)
print('\n>>> Tahap 4 SELESAI. Lanjut ke 05_evaluation.ipynb')